In [1]:
import pandas as pd
import requests

# ══════════════════════════════════════════════
#  CONFIG — ปรับได้ตามต้องการ
# ══════════════════════════════════════════════
API_URL         = "http://localhost:8003/analyze"  # ms3 โดยตรง (ไม่ผ่าน ms2)
WINDOW_SIZE     = 96    # rows ต่อ window (96 × 10s ≈ 16 นาที)
STEP_SIZE       = 96    # 96 = non-overlapping (เร็ว) | 24 = overlapping (ข้อมูลหนาแน่น)
REQUEST_TIMEOUT = 300   # วินาที — Chronos run 9 sensor ต่อ request
MAX_RETRIES     = 2

ALL_SENSORS = [
    "temperature_c", "vibration_rms", "rpm",
    "pressure_bar", "flow_lpm", "current_a",
    "oil_temp_c", "humidity_pct", "power_kw",
]

# ══════════════════════════════════════════════
#  โหลดข้อมูล
# ══════════════════════════════════════════════
df = pd.read_csv("telemetry_data.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values(["device_id", "timestamp"]).reset_index(drop=True)

devices = sorted(df["device_id"].unique().tolist())
rows_per_device = {d: len(df[df["device_id"] == d]) for d in devices}

print(f"Loaded {len(df):,} rows | {len(devices)} devices")
for d, n in rows_per_device.items():
    wins = max(0, (n - WINDOW_SIZE) // STEP_SIZE + 1)
    print(f"  {d}: {n} rows → {wins} windows")

min_rows      = min(rows_per_device.values())
total_windows = max(0, (min_rows - WINDOW_SIZE) // STEP_SIZE + 1)
print(f"\n→ จะส่ง {total_windows} API calls (ทุก device พร้อมกันใน 1 request)")
print(f"→ เวลาโดยประมาณ: {total_windows * 2}–{total_windows * 5} นาที")


Loaded 33,660 rows | 5 devices
  boiler-feed-401: 6732 rows → 70 windows
  cnc-spindle-301: 6732 rows → 70 windows
  filling-comp-201: 6732 rows → 70 windows
  mix-pump-101: 6732 rows → 70 windows
  pack-conveyor-501: 6732 rows → 70 windows

→ จะส่ง 70 API calls (ทุก device พร้อมกันใน 1 request)
→ เวลาโดยประมาณ: 140–350 นาที


In [2]:
# ══════════════════════════════════════════════
#  Health Check — ต้องผ่านก่อนรัน Cell ถัดไป
# ══════════════════════════════════════════════
try:
    r = requests.get("http://localhost:8003/health", timeout=5)
    r.raise_for_status()
    print(f"ms3-ai-engine: {r.json()}")
except Exception as e:
    print(f"ms3-ai-engine ไม่ตอบสนอง: {e}")
    print("รัน: docker compose up -d ms3-ai-engine ms3-worker")
    raise SystemExit("หยุด: ต้องให้ ms3 ทำงานก่อน")


ms3-ai-engine: {'status': 'ok', 'service': 'ms3-ai-engine', 'mode': 'web'}


In [ ]:

# ══════════════════════════════════════════════
#  Sliding Window Loop — เก็บ anomaly scores
# ══════════════════════════════════════════════
import json
import numpy as np

results     = []
device_dfs  = {d: df[df["device_id"] == d].reset_index(drop=True) for d in devices}

def get_risk(score):
    if score >= 0.75: return "critical"
    if score >= 0.50: return "high"
    if score >= 0.30: return "medium"
    return "low"

for window_idx in range(total_windows):
    start = window_idx * STEP_SIZE
    end   = start + WINDOW_SIZE

    # รวมทุก device สำหรับ window นี้ → ส่ง 1 request เดียว
    # ใช้ to_json + json.loads เพื่อแปลง Timestamp/numpy types ให้ JSON serializable
    window_rows = []
    for device_id, ddf in device_dfs.items():
        if end > len(ddf):
            continue
        chunk = ddf.iloc[start:end]
        records = json.loads(chunk.to_json(orient="records", date_format="iso", date_unit="s"))
        window_rows.extend(records)

    if not window_rows:
        continue

    ref_df    = device_dfs[devices[0]].iloc[start:end]
    win_start = str(ref_df["timestamp"].iloc[0])
    win_end   = str(ref_df["timestamp"].iloc[-1])

    print(f"[{window_idx+1}/{total_windows}] {win_start[:19]} → {win_end[:19]}", end="  ")

    # ── ส่ง request (พร้อม retry) ──────────────────────────────
    payload       = {"telemetry": window_rows}
    response_data = None

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            resp = requests.post(API_URL, json=payload, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            response_data = resp.json()
            break
        except requests.exceptions.Timeout:
            if attempt <= MAX_RETRIES:
                print(f"(timeout, retry {attempt})", end="  ")
            else:
                print("(timeout หมด retry)")
        except requests.exceptions.RequestException as e:
            print(f"(error: {e})")
            break

    if response_data is None:
        print("→ skip")
        continue

    # ── บันทึกผลรายเครื่อง ──────────────────────────────────────
    per_device    = response_data.get("per_device") or {}
    overall_score = response_data.get("anomaly_score", 0.0)
    overall_risk  = response_data.get("risk_level", "unknown")

    for device_id in devices:
        dev_result    = per_device.get(device_id, {})
        dev_score     = dev_result.get("anomaly_score", 0.0)
        sensor_scores = dev_result.get("per_sensor", {})

        row = {
            "window_id":     window_idx + 1,
            "device_id":     device_id,
            "window_start":  win_start,
            "window_end":    win_end,
            "anomaly_score": dev_score,
            "risk_level":    get_risk(dev_score),
        }
        for s in ALL_SENSORS:
            row[f"{s}_score"] = sensor_scores.get(s, None)

        results.append(row)

    print(f"score={overall_score:.4f}  {overall_risk}")

print(f"\nเสร็จ! เก็บได้ {len(results)} rows ({len(results)//max(len(devices),1)} windows × {len(devices)} devices)")


[1/70] 2026-03-15 10:04:40 → 2026-03-15 10:35:10  score=1.0000  critical
[2/70] 2026-03-15 10:35:20 → 2026-03-15 11:07:01  score=1.0000  critical
[3/70] 2026-03-15 11:07:21 → 2026-03-15 11:39:30  score=1.0000  critical
[4/70] 2026-03-15 11:39:50 → 2026-03-15 12:16:30  score=1.0000  critical
[5/70] 2026-03-15 12:17:00 → 2026-03-15 12:48:01  score=1.0000  critical
[6/70] 2026-03-15 12:48:21 → 2026-03-15 13:21:11  score=1.0000  critical
[7/70] 2026-03-15 13:21:21 → 2026-03-15 13:56:01  score=1.0000  critical
[8/70] 2026-03-15 13:56:11 → 2026-03-15 14:29:31  score=1.0000  critical
[9/70] 2026-03-15 14:29:51 → 2026-03-15 14:59:11  score=1.0000  critical
[10/70] 2026-03-15 14:59:21 → 2026-03-15 15:30:31  score=1.0000  critical
[11/70] 2026-03-15 15:31:01 → 2026-03-15 16:05:01  score=1.0000  critical
[12/70] 2026-03-15 16:05:31 → 2026-03-15 16:36:31  score=1.0000  critical
[13/70] 2026-03-15 16:36:41 → 2026-03-15 17:09:21  score=1.0000  critical
[14/70] 2026-03-15 17:09:41 → 2026-03-15 17:42:

In [ ]:
# ══════════════════════════════════════════════
#  บันทึก training_windows.csv
# ══════════════════════════════════════════════
if not results:
    print("ไม่มีข้อมูล — รัน Cell 3 ใหม่")
else:
    df_out = pd.DataFrame(results)
    df_out.to_csv("training_windows.csv", index=False)

    print(f"บันทึก training_windows.csv แล้ว  ({len(df_out):,} rows × {len(df_out.columns)} columns)")
    print(f"\nสถิติ anomaly_score รายเครื่อง:")
    print(df_out.groupby("device_id")["anomaly_score"].describe().round(4).to_string())
    print(f"\nการกระจาย risk_level:")
    print(df_out["risk_level"].value_counts().to_string())
    print(f"\nตัวอย่าง 3 แถวแรก:")
    cols = ["window_id","device_id","anomaly_score","risk_level","temperature_c_score","vibration_rms_score"]
    print(df_out.head(3)[cols].to_string())
